# Converting a trained model to tflite
https://www.tensorflow.org/lite/microcontrollers/build_convert#model_conversion

# Convert model to tflite

In [1]:
import tensorflow as tf
import numpy as np

2026-04-16 15:39:02.163748: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 15:39:02.298191: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 15:39:02.793848: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/cluster/.conda/envs/tf210/lib:
2026-04-16 15:39:02.793927: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin

In [2]:
training_spectrogram = np.load('training_spectrogram.npz')
validation_spectrogram = np.load('validation_spectrogram.npz')
test_spectrogram = np.load('test_spectrogram.npz')

X_train = training_spectrogram['X']
X_validate = validation_spectrogram['X']
X_test = test_spectrogram['X']

complete_train_X = np.concatenate((X_train, X_validate, X_test))

In [5]:
converter2 = tf.lite.TFLiteConverter.from_saved_model("fully_trained.model")
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
def representative_dataset_gen():
    for i in range(0, len(complete_train_X), 100):
        # Get sample input data as a numpy array in a method of your choosing.
        yield [complete_train_X[i:i+100]]
converter2.representative_dataset = representative_dataset_gen
# converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
tflite_quant_model = converter2.convert()
open("converted_model.tflite", "wb").write(tflite_quant_model)

2026-04-16 15:42:17.449760: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2026-04-16 15:42:17.449793: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2026-04-16 15:42:17.449958: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: fully_trained.model
2026-04-16 15:42:17.452148: I tensorflow/cc/saved_model/reader.cc:89] Reading meta graph with tags { serve }
2026-04-16 15:42:17.452164: I tensorflow/cc/saved_model/reader.cc:130] Reading SavedModel debug info (if present) from: fully_trained.model
2026-04-16 15:42:17.457593: I tensorflow/cc/saved_model/loader.cc:229] Restoring SavedModel bundle.
2026-04-16 15:42:17.505917: I tensorflow/cc/saved_model/loader.cc:213] Running initialization op on SavedModel bundle at path: fully_trained.model
2026-04-16 15:42:17.516335: I tensorflow/cc/saved_model/loader.cc:305] SavedModel load for tags { serve }; Status: success: OK. Too

43416

# To convert to C++
This will run a command line too to convert out tflite model into C code.

In [4]:
!xxd -i converted_model.tflite > model_data.cc

/bin/bash: /home/cluster/.conda/envs/tf210/lib/libtinfo.so.6: no version information available (required by /bin/bash)
